# 📚 Módulo 01 - MLOps y CI/CD para Machine Learning

## 🎯 Objetivos

1. ✅ Entender qué es MLOps y su importancia
2. ✅ Implementar CI/CD para modelos ML
3. ✅ Versionar datos, código y modelos
4. ✅ Automatizar pipelines de entrenamiento y despliegue

---

## 1️⃣ ¿Qué es MLOps?

**MLOps** = Machine Learning + DevOps

Combina prácticas de:
* **Development (Dev)**: Desarrollo de modelos, experimentación
* **Operations (Ops)**: Despliegue, monitoreo, escalabilidad
* **Machine Learning**: Datos, features, entrenamiento

### Ciclo de Vida ML

```
1. Problem Definition → 2. Data Collection
         ↓                        ↓
    9. Monitor          ← 3. Feature Engineering
         ↓                        ↓
    8. Deploy          ← 4. Model Training
         ↓                        ↓
    7. Validate        ← 5. Evaluation
         ↓                        ↓
    6. Test            ← Loop until acceptable
```

**MLOps automatiza y estandariza este ciclo**

---

## 2️⃣ Componentes de MLOps

### Versionado

**Código (Git)**:
```bash
git commit -m "Add feature: rolling_avg"
git push origin main
```

**Datos (DVC - Data Version Control)**:
```bash
dvc add data/train.csv
dvc push
```

**Modelos (MLflow)**:
```python
import mlflow

with mlflow.start_run():
    mlflow.log_params({"max_depth": 5, "n_estimators": 100})
    mlflow.sklearn.log_model(model, "random_forest")
    mlflow.log_metric("accuracy", 0.92)
```

### Tracking de Experimentos

```python
import mlflow

mlflow.set_experiment("churn_prediction")

for lr in [0.01, 0.001, 0.0001]:
    with mlflow.start_run():
        model = train_model(learning_rate=lr)
        accuracy = evaluate(model, X_test, y_test)
        
        mlflow.log_param("learning_rate", lr)
        mlflow.log_metric("accuracy", accuracy)
        mlflow.sklearn.log_model(model, "model")
```

### Registry de Modelos

```python
# Registrar modelo
mlflow.register_model(
    model_uri="runs:/<run_id>/model",
    name="churn_predictor"
)

# Transicionar a producción
client = mlflow.tracking.MlflowClient()
client.transition_model_version_stage(
    name="churn_predictor",
    version=3,
    stage="Production"
)
```

**Stages**:
* **None**: Modelo recién registrado
* **Staging**: En pruebas
* **Production**: Sirviendo tráfico real
* **Archived**: Modelo antiguo

---

## 3️⃣ CI/CD para Machine Learning

### Continuous Integration (CI)

**Objetivo**: Validar cambios automáticamente

```yaml
# .github/workflows/ci.yml
name: CI Pipeline

on: [push, pull_request]

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v2
      
      - name: Set up Python
        uses: actions/setup-python@v2
        with:
          python-version: 3.9
      
      - name: Install dependencies
        run: pip install -r requirements.txt
      
      - name: Run unit tests
        run: pytest tests/
      
      - name: Check data quality
        run: python scripts/validate_data.py
      
      - name: Validate model performance
        run: python scripts/test_model.py
```

**Tests típicos**:
```python
import pytest

def test_data_schema():
    df = load_data()
    assert "customer_id" in df.columns
    assert df["age"].dtype == "int64"

def test_no_data_leakage():
    train, test = split_data()
    assert set(train.index).isdisjoint(set(test.index))

def test_model_accuracy():
    model = load_model()
    accuracy = evaluate(model, X_test, y_test)
    assert accuracy >= 0.85  # threshold mínimo
```

### Continuous Deployment (CD)

**Objetivo**: Desplegar automáticamente modelos validados

```yaml
# .github/workflows/cd.yml
name: CD Pipeline

on:
  push:
    branches: [main]

jobs:
  deploy:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v2
      
      - name: Train model
        run: python scripts/train.py
      
      - name: Evaluate model
        run: python scripts/evaluate.py
      
      - name: Register model in MLflow
        run: python scripts/register_model.py
      
      - name: Deploy to staging
        run: |
          databricks jobs run-now --job-id $STAGING_JOB_ID
      
      - name: Run smoke tests
        run: python scripts/smoke_test.py
      
      - name: Promote to production
        if: success()
        run: python scripts/promote_to_prod.py
```

---

## 4️⃣ Pipelines Automatizados

### Pipeline de Entrenamiento

```python
from databricks import workflows

# Definir pipeline
pipeline = workflows.Pipeline(
    name="train_churn_model",
    schedule="@daily",
    tasks=[
        # Task 1: Preparar datos
        {
            "task_key": "prepare_data",
            "notebook_path": "/pipelines/01_prepare_data",
            "cluster": {"num_workers": 2}
        },
        # Task 2: Feature engineering
        {
            "task_key": "features",
            "notebook_path": "/pipelines/02_features",
            "depends_on": [{"task_key": "prepare_data"}]
        },
        # Task 3: Entrenar modelo
        {
            "task_key": "train",
            "notebook_path": "/pipelines/03_train",
            "depends_on": [{"task_key": "features"}]
        },
        # Task 4: Evaluar y registrar
        {
            "task_key": "evaluate",
            "notebook_path": "/pipelines/04_evaluate",
            "depends_on": [{"task_key": "train"}]
        }
    ]
)
```

### Reentrenamiento Automático

**Triggers**:
1. **Schedule**: Cada N días/semanas
2. **Performance degradation**: Accuracy < threshold
3. **Data drift**: Distribución de features cambió
4. **Manual**: Usuario solicita reentrenamiento

```python
# Monitoreo de performance
from mlflow import MlflowClient

client = MlflowClient()
model = client.get_latest_versions("churn_predictor", stages=["Production"])[0]

# Evaluar en datos recientes
current_accuracy = evaluate_production_model(model)

if current_accuracy < 0.85:
    print("⚠️ Performance degradation detected")
    trigger_retraining_pipeline()
```

---

## 5️⃣ Infraestructura como Código (IaC)

### Terraform para Databricks

```hcl
# main.tf
resource "databricks_cluster" "ml_cluster" {
  cluster_name = "ml-training-cluster"
  spark_version = "11.3.x-cpu-ml-scala2.12"
  node_type_id = "i3.xlarge"
  num_workers = 2
  
  autoscale {
    min_workers = 1
    max_workers = 4
  }
}

resource "databricks_job" "train_model" {
  name = "train_churn_model"
  
  task {
    task_key = "train"
    notebook_task {
      notebook_path = "/pipelines/train"
    }
    existing_cluster_id = databricks_cluster.ml_cluster.id
  }
  
  schedule {
    quartz_cron_expression = "0 0 2 * * ?"  # 2 AM diario
    timezone_id = "America/Argentina/Buenos_Aires"
  }
}
```

---

## 6️⃣ Mejores Prácticas MLOps

### ✅ DO

1. **Versionar todo**: Código, datos, modelos, config
2. **Automatizar**: CI/CD, tests, despliegues
3. **Monitorear**: Performance, latencia, drift
4. **Documentar**: Experimentos, decisiones, resultados
5. **Testing**: Unit tests, integration tests, A/B tests
6. **Reproducibilidad**: Seeds fijos, ambientes idénticos
7. **Rollback fácil**: Mantener versiones anteriores

### ❌ DON'T

1. No entrenar en producción sin validación
2. No desplegar sin smoke tests
3. No hardcodear configuraciones
4. No ignorar data quality checks
5. No olvidar monitorear post-deployment
6. No mezclar ambientes (dev, staging, prod)

---

## 7️⃣ Herramientas del Ecosistema MLOps

| Categoría | Herramientas |
|-----------|-------------|
| **Tracking** | MLflow, Weights & Biases, Neptune.ai |
| **Versionado Datos** | DVC, LakeFS, Pachyderm |
| **CI/CD** | GitHub Actions, GitLab CI, Jenkins |
| **Orquestación** | Airflow, Prefect, Databricks Workflows |
| **Serving** | MLflow Models, SageMaker, Seldon |
| **Monitoring** | Evidently AI, Arize, WhyLabs |
| **IaC** | Terraform, Pulumi, CloudFormation |

---

## ✅ Checklist de Madurez MLOps

**Nivel 0 - Manual**:
- [ ] Entrenamientos manuales
- [ ] Sin versionado de modelos
- [ ] Deployment manual

**Nivel 1 - Automatización Básica**:
- [ ] Scripts de entrenamiento
- [ ] Versionado con MLflow
- [ ] CI/CD básico

**Nivel 2 - Automatización Completa**:
- [ ] Pipelines end-to-end
- [ ] Reentrenamiento automático
- [ ] Monitoring de drift
- [ ] A/B testing

**Nivel 3 - MLOps Maduro**:
- [ ] Feature stores
- [ ] Canary deployments
- [ ] Auto-scaling
- [ ] FinOps (cost optimization)

---

**Universidad del Aconcagua 🇦🇷**